# Домашнє завдання 3 — Event Sourcing: ShoppingCart

Реалізація Event Sourcing для кошика покупок з анімованою діаграмою потоку подій та змін стану.

## 1. Визначення подій та Value Object

In [1]:
from __future__ import annotations
from dataclasses import dataclass, field
from decimal import Decimal
from uuid import UUID, uuid4
from functools import singledispatchmethod


class CartError(Exception):
    """Domain exception for shopping cart invariant violations."""


# --- Events ---

@dataclass(frozen=True)
class ItemAdded:
    product_id: str
    product_name: str
    price: Decimal
    quantity: int

@dataclass(frozen=True)
class ItemRemoved:
    product_id: str

@dataclass(frozen=True)
class DiscountApplied:
    discount_percent: Decimal

@dataclass(frozen=True)
class CartCheckedOut:
    total_amount: Decimal

type Event = ItemAdded | ItemRemoved | DiscountApplied | CartCheckedOut


# --- Value Object (immutable) ---

@dataclass(frozen=True)
class CartItem:
    product_id: str
    product_name: str
    price: Decimal
    quantity: int

print("Events & CartItem defined")

Events & CartItem defined


## 2. Агрегат ShoppingCart

In [2]:
@dataclass
class ShoppingCart:
    cart_id: UUID = field(default_factory=uuid4)
    items: dict[str, CartItem] = field(default_factory=dict)
    discount_percent: Decimal = Decimal(0)
    total_amount: Decimal | None = None
    is_checked_out: bool = False
    version: int = 0
    _pending_events: list[Event] = field(default_factory=list, repr=False)

    @property
    def pending_events(self) -> tuple[Event, ...]:
        return tuple(self._pending_events)

    def clear_pending_events(self) -> None:
        self._pending_events.clear()

    # --- Commands ---

    def add_item(self, product_id: str, name: str, price: Decimal, quantity: int) -> None:
        if self.is_checked_out:
            raise CartError("Кошик вже оформлено")
        if quantity <= 0:
            raise CartError("Кількість має бути > 0")
        if price < 0:
            raise CartError("Ціна не може бути від\'ємною")
        if product_id in self.items and self.items[product_id].price != price:
            raise CartError(
                f"Товар {product_id} вже в кошику з ціною {self.items[product_id].price}, "
                f"отримано {price}"
            )
        self._raise(ItemAdded(product_id, name, price, quantity))

    def remove_item(self, product_id: str) -> None:
        if self.is_checked_out:
            raise CartError("Кошик вже оформлено")
        if product_id not in self.items:
            raise CartError("Товар не в кошику")
        self._raise(ItemRemoved(product_id))

    def apply_discount(self, percent: Decimal) -> None:
        if self.is_checked_out:
            raise CartError("Кошик вже оформлено")
        if not (Decimal(0) < percent <= Decimal(50)):
            raise CartError("Знижка має бути від 0% до 50%")
        if self.discount_percent > 0:
            raise CartError("Знижку вже застосовано")
        self._raise(DiscountApplied(percent))

    def checkout(self) -> None:
        if self.is_checked_out:
            raise CartError("Кошик вже оформлено")
        if not self.items:
            raise CartError("Кошик порожній")
        total = sum(i.price * i.quantity for i in self.items.values())
        total *= (1 - self.discount_percent / 100)
        self._raise(CartCheckedOut(total))

    # --- Apply (state mutation only) ---

    @singledispatchmethod
    def _apply(self, event: Event) -> None:
        raise TypeError(f"Unknown event: {type(event)}")

    @_apply.register
    def _(self, e: ItemAdded) -> None:
        if e.product_id in self.items:
            existing = self.items[e.product_id]
            self.items[e.product_id] = CartItem(
                existing.product_id, existing.product_name,
                existing.price, existing.quantity + e.quantity,
            )
        else:
            self.items[e.product_id] = CartItem(
                e.product_id, e.product_name, e.price, e.quantity,
            )

    @_apply.register
    def _(self, e: ItemRemoved) -> None:
        del self.items[e.product_id]

    @_apply.register
    def _(self, e: DiscountApplied) -> None:
        self.discount_percent = e.discount_percent

    @_apply.register
    def _(self, e: CartCheckedOut) -> None:
        self.is_checked_out = True
        self.total_amount = e.total_amount

    # --- Infrastructure ---

    def _raise(self, event: Event) -> None:
        self._apply(event)
        self.version += 1
        self._pending_events.append(event)

    @classmethod
    def from_history(cls, cart_id: UUID, events: list[Event]) -> ShoppingCart:
        cart = cls(cart_id=cart_id)
        for event in events:
            cart._apply(event)
            cart.version += 1
        return cart

print("ShoppingCart aggregate defined")

ShoppingCart aggregate defined


## 3. Тест (Вимога 5)

Створити кошик, додати 3 товари, видалити один, застосувати знижку, оформити замовлення.
Перевірити відповідність стану + replay через `from_history`.

In [3]:
cart = ShoppingCart()

cart.add_item("p1", "Молоко", Decimal("32.50"), 2)
cart.add_item("p2", "Хліб", Decimal("18.00"), 1)
cart.add_item("p3", "Масло", Decimal("65.00"), 1)
cart.remove_item("p2")
cart.apply_discount(Decimal("10"))
cart.checkout()

assert len(cart.items) == 2
assert "p1" in cart.items and "p3" in cart.items and "p2" not in cart.items
assert cart.discount_percent == Decimal("10")
assert cart.is_checked_out is True
assert cart.total_amount == Decimal("117.000")  # (32.50*2 + 65)*0.9
assert cart.version == 6

# Replay
restored = ShoppingCart.from_history(cart.cart_id, list(cart.pending_events))
assert restored.items == cart.items
assert restored.discount_percent == cart.discount_percent
assert restored.is_checked_out == cart.is_checked_out
assert restored.total_amount == cart.total_amount
assert restored.version == cart.version

print("All assertions passed")
print(f"Events: {len(cart.pending_events)}, Version: {cart.version}")
print(f"Total: {cart.total_amount}")

All assertions passed
Events: 6, Version: 6
Total: 117.000


## 4. Перевірка бізнес-правил

In [4]:
errors_caught = []

def expect_error(label, fn):
    try:
        fn()
        errors_caught.append(f"FAIL: {label} — no exception")
    except CartError as e:
        errors_caught.append(f"OK:   {label} — {e}")

# Від'ємна кількість
expect_error("qty <= 0", lambda: ShoppingCart().add_item("x", "X", Decimal("10"), -1))

# Видалення неіснуючого
expect_error("remove missing", lambda: ShoppingCart().remove_item("ghost"))

# Знижка > 50%
expect_error("discount > 50", lambda: ShoppingCart().apply_discount(Decimal("51")))

# Подвійна знижка
def double_discount():
    c = ShoppingCart()
    c.apply_discount(Decimal("10"))
    c.apply_discount(Decimal("20"))
expect_error("double discount", double_discount)

# Порожній checkout
expect_error("empty checkout", lambda: ShoppingCart().checkout())

# Подвійний checkout
def double_checkout():
    c = ShoppingCart()
    c.add_item("p1", "X", Decimal("10"), 1)
    c.checkout()
    c.checkout()
expect_error("double checkout", double_checkout)

# Конфлікт цін
def price_conflict():
    c = ShoppingCart()
    c.add_item("p1", "X", Decimal("10"), 1)
    c.add_item("p1", "X", Decimal("15"), 1)
expect_error("price conflict", price_conflict)

for line in errors_caught:
    print(line)

OK:   qty <= 0 — Кількість має бути > 0
OK:   remove missing — Товар не в кошику
OK:   discount > 50 — Знижка має бути від 0% до 50%
OK:   double discount — Знижку вже застосовано
OK:   empty checkout — Кошик порожній
OK:   double checkout — Кошик вже оформлено
OK:   price conflict — Товар p1 вже в кошику з ціною 10, отримано 15


## Архітектура

Кожна **команда** (add_item, remove_item, apply_discount, checkout) проходить валідацію бізнес-правил,
генерує іммутабельну **подію**, яка застосовується до стану через `_apply` (singledispatch).
Метод `from_history` відтворює агрегат із послідовності подій — replay без повторної валідації.

Потік: `Command → validate → Event → _apply(state) → pending_events[]`